In [ ]:
import requests
from langchain.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import CharacterTextSplitter
from dotenv import load_dotenv, find_dotenv 
from langchain.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain.prompts import ChatPromptTemplate
from langchain.chat_models import ChatOpenAI
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser

from download_cards import download_model_cards

from retriever import Retriever
from generator import Generator

_ = load_dotenv(find_dotenv())

In [ ]:
download_model_cards()

# Indexing 

In [ ]:

# Step 1.1: load documents
loader = DirectoryLoader('model_cards/', glob="**/*.md", loader_cls=TextLoader)
documents = loader.load()
# Step 1.2: split documents into chunks
text_splitter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = text_splitter.split_documents(documents)

In [ ]:
model_name = "sentence-transformers/all-mpnet-base-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}
embeddings = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)
# Step 1.3: encode chunks into vectors and store in a vector database
vectordb = FAISS.from_documents(documents, embeddings)


# Retrieval

In [ ]:
# Step 2: Retrieval: retrieve the Top k chunks most relevant to the question based on semantic similarity.
retriever = vectordb.as_retriever(
    search_type="similarity",
    search_kwargs={'k': 5}
)

# Generation

In [ ]:
generator = Generator(model="gpt-4o")
template = generator.format_prompt(
    system_prompt_path="prompts/system_prompt_qa.txt", 
    user_prompt_path="prompts/user_prompt_qa.txt",
    card_id="123",
    section="test_case",
    )
template

## Prompt Engineering
- TODO: DSPy

In [ ]:
#template = """You are an assistant for question-answering tasks. 
#Use the following pieces of retrieved context to answer the question. 
#If you don't know the answer, just say that you don't know. 
#Use three sentences maximum and keep the answer concise.
#Question: {question} 
#Context: {context} 
#Answer:
#"""

prompt = ChatPromptTemplate.from_template(template)

print(prompt)

In [ ]:
# Step 3: Generation: input the original question and the retrieved chunks together into LLM to generate the final answer.
llm = ChatOpenAI(model_name="gpt-4o", temperature=0.5)

rag_chain = (
    {"context": retriever,  "question": RunnablePassthrough()} 
    | prompt 
    | llm
    | StrOutputParser() 
)

query = "Please generate a new python script that detects data drift for tabular data?"
rag_chain.invoke(query)

In [ ]:
!ls ..

# RAG Evaluation

In [ ]:
# Import necessary libraries
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# Instantiate the models
generator_llm = ChatOpenAI(model="gpt-4o-mini")
critic_llm = ChatOpenAI(model="gpt-4o-mini")
embeddings = OpenAIEmbeddings()

# Create the TestsetGenerator
generator = TestsetGenerator.from_langchain(
    generator_llm,
    critic_llm,
    embeddings
)

# Call the generator
testset = generator.generate_with_langchain_docs(
data_transformed, 
test_size=20, 
distributions={ 
simple: 0.5, 
reasoning: 0.25, 
multi_context: 0.25}
)